# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook chạy **toàn bộ** quy trình bằng mã trong thư mục `code/` (cùng thư mục với notebook này). Cần **GPU** (Kaggle: bật *Internet* và GPU P100/T4 trong Settings).

**Cách dùng:** chạy từ trên xuống; mỗi giai đoạn **idempotent**: lần chạy đã có `summary.json` được bỏ qua, phiên bị ngắt thì chạy lại ô đó là tiếp tục (resume từ `last.pt`). Mọi quyết định chọn cấu hình dùng **val** và được ghi vào `runs/decisions.json`. **Test chỉ được mở ở ô "Bước 4"**, mỗi seed đúng một lần (có chốt chặn).

| Giai đoạn | Ô | Kết quả |
|---|---|---|
| 0 | Cài đặt, dữ liệu, EDA, kiểm tra pipeline | `figures/` |
| 1 | So sánh backbone `B01-B07`, chọn backbone | `runs/B0x`, `decisions.json` |
| 2 | Nhiễu `T00` x3 seed, yếu tố `T01-T13`, kết hợp `T20`, chốt công thức | `runs/T*`, `curves/` |
| 3 | Suy luận trên val: `I00-I08`, độ trễ | `runs/inference/`, `figures/` |
| 4 | Chung kết `F01` x3 seed, test một lần, chấm bằng `eval.py` | `predictions/`, `eval_out/` |
| 5 | `results.xlsx`, kiểm tra nhất quán, đóng gói | thư mục nộp bài |

## 0. Cài đặt môi trường

In [ ]:
# Kaggle: bật Internet trong Settings. Colab: chạy bình thường.
!pip -q install timm openpyxl

import os, sys, platform, shutil, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# ---- repo và thư mục bài làm ----
REPO_URL = "https://github.com/DuyNguyen6868/K4-Track4-Day2-Deeplearning-Advance.git"   # repo đã push
SUB_NAME = "2A202602815_nguyen_canh_duy"
REPO_BRANCH = "submission/2A202602815"   # nhánh chứa submissions/<SUB_NAME>/code
# ------------------------

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
os.chdir(WORK)
REPO_DIR = WORK / "K4-Track4-Day2-Deeplearning-Advance"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
SUB_DIR = REPO_DIR / "submissions" / SUB_NAME
CODE_DIR = SUB_DIR / "code"
assert (CODE_DIR / "train.py").exists(), f"Không thấy {CODE_DIR}: kiểm tra REPO_URL và SUB_NAME"
assert (REPO_DIR / "eval.py").exists(), "Thiếu eval.py gốc ở gốc repo (eval.py không được sửa)"
sys.path.insert(0, str(CODE_DIR))
os.environ["LAB_EVAL_PY"] = str(REPO_DIR / "eval.py")

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU: huấn luyện thật cần GPU")

### Khôi phục kết quả của phiên trước (tuỳ chọn)
Kaggle chỉ giữ `/kaggle/working` của phiên hiện tại. Nếu bạn đã lưu `runs/`, `curves/`, `predictions/` thành một Kaggle Dataset ở phiên trước, đặt đường dẫn vào `RESTORE_FROM` để tiếp tục thay vì chạy lại.

In [ ]:
RESTORE_FROM = ""   # ví dụ "/kaggle/input/lab-day2-runs"; để trống nếu chạy mới
if RESTORE_FROM and Path(RESTORE_FROM).exists():
    for name in ("runs", "curves", "predictions", "figures"):
        src = Path(RESTORE_FROM) / name
        if src.exists() and not (WORK / name).exists():
            shutil.copytree(src, WORK / name)
            print("đã khôi phục", name)

### Tải dữ liệu
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và file chia fold từ GitHub của tác giả DeepWeeds. **Không sửa, không chia lại** các file CSV.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"data/labels/{name}.csv"):
        !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
# Tự tìm thư mục chứa ảnh .jpg sau khi giải nén
IMAGES_DIR = str(next(Path("data").rglob("*.jpg")).parent)
LABELS_DIR = "data/labels"
N_IMG = sum(1 for _ in Path(IMAGES_DIR).glob("*.jpg"))
print("IMAGES_DIR =", IMAGES_DIR, "| số ảnh:", N_IMG)
assert N_IMG == 17509, "Số ảnh phải là 17509"

# Tuỳ chọn dùng chung cho mọi lệnh CLI của code/
COMMON = f"--images-dir {IMAGES_DIR} --labels-dir {LABELS_DIR} --out-dir runs --pred-dir predictions --curves-dir curves --num-workers {min(4, os.cpu_count() or 2)}"

## Bước 0 — EDA và kiểm tra pipeline
Kiểm tra chia dữ liệu (không giao nhau, tổng 17509, 60/20/20, đủ file), phân phối lớp, ≥ 3 ảnh mỗi lớp, đối chiếu Table 1 của bài báo. Rồi kiểm tra pipeline: loss ban đầu ≈ ln 9 = 2,197, overfit một batch, xem ảnh sau augmentation.

In [ ]:
import eda
eda_out = eda.run_eda(IMAGES_DIR, LABELS_DIR, "figures")
from IPython.display import Image, display
display(Image("figures/class_distribution.png"))
display(Image("figures/samples_per_class.png"))

In [ ]:
# Backbone dùng để kiểm tra: resnet50 (cùng công thức nền). Cần tải trọng số timm (Internet bật).
san = eda.run_sanity("resnet50", IMAGES_DIR, LABELS_DIR, "figures")
assert san["ok"], "Pipeline chưa ổn (loss ban đầu hoặc overfit một batch): dừng lại và kiểm tra trước khi train"
display(Image("figures/augmentation_preview.png"))

## Bước 1 — So sánh backbone (7 mạng, 4 họ)
Cùng công thức nền `T00`, cùng seed 0, mỗi backbone một lần chạy (`B01-B07`), trọng số **ghim tag** để so sánh công bằng. Quy tắc chọn (đặt trước khi xem kết quả, ghi trong `code/experiments.py`): các mạng có macro-F1 val ≥ (tốt nhất − 0,003) là ứng viên; chọn mạng có độ trễ batch-1 thấp nhất trong số đó.

> Tốn nhiều GPU nhất của bài lab. Nếu hết giờ GPU giữa chừng, chạy lại ô này ở phiên sau (resume). Hạ ngân sách bằng `--epochs N` thì **phải ghi vào báo cáo**.

In [ ]:
!python {CODE_DIR}/experiments.py {COMMON} list

In [ ]:
!python {CODE_DIR}/experiments.py {COMMON} backbones

In [ ]:
!python {CODE_DIR}/experiments.py {COMMON} select-backbone
print(json.dumps({k: v for k, v in json.load(open("runs/decisions.json", encoding="utf-8")).items() if k.startswith("backbone")}, indent=2, ensure_ascii=False))

## Bước 2 — Công thức huấn luyện
1. `noise`: `T00` chạy 3 seed để đo nhiễu (`std_T00`, ddof=1).
2. `training`: `T01-T13` (mỗi lần chỉ khác `T00` một yếu tố; trục A-G). Thêm `--include-extra` cho `T14-T19` nếu còn giờ GPU.
3. `winners`: yếu tố thắng nếu Δ ≥ max(2·std, 0,003). `combo`: ghép yếu tố thắng tốt nhất mỗi ô (`T20`). `recipe`: chốt công thức cuối bằng val.

In [ ]:
!python {CODE_DIR}/experiments.py {COMMON} noise

In [ ]:
!python {CODE_DIR}/experiments.py {COMMON} training

In [ ]:
# Tuỳ chọn (còn giờ GPU): T14-T19
# !python {CODE_DIR}/experiments.py {COMMON} training --include-extra

In [ ]:
!python {CODE_DIR}/experiments.py {COMMON} winners
!python {CODE_DIR}/experiments.py {COMMON} combo
!python {CODE_DIR}/experiments.py {COMMON} recipe

In [ ]:
# Huấn luyện lại cấu hình chung kết F01 (3 seed). Dùng lại lần chạy giống hệt qua alias nếu có (ghi rõ trong xlsx).
!python {CODE_DIR}/experiments.py {COMMON} final-train

## Bước 3 — Suy luận trên val và độ trễ
Dùng `F01` seed 0, **chỉ val**. Gồm `I00` (mốc), TTA lật/5-crop/đa tỉ lệ, gộp xác suất vs logit, độ phân giải kiểm tra, ensemble, EMA vs thô, temperature scaling, gộp BN, FP16/AMP. Độ trễ đo đúng cách (warmup ≥ 10, synchronize, ≥ 50 lần, p50/p95/p99). Quyết định bộ view cho chung kết và cấu hình thời gian thực (p95 ≤ 100 ms) được ghi vào `decisions.json`.

In [ ]:
!python {CODE_DIR}/study.py --images-dir {IMAGES_DIR} --labels-dir {LABELS_DIR} --out-dir runs

In [ ]:
t = pd.read_csv("runs/inference/inference_table.csv")
display(t[["exp_id", "method", "K", "macro_f1_val", "top1_val", "ece_val", "p50_ms", "p95_ms", "cost_vs_I00"]].round(4))
display(Image("figures/tradeoff_f1_vs_latency.png"))
display(Image("figures/reliability_val.png")) if Path("figures/reliability_val.png").exists() else None
print(json.dumps({k: v for k, v in json.load(open("runs/decisions.json", encoding="utf-8")).items() if k in ("inference", "realtime")}, indent=2, ensure_ascii=False))

## Bước 4 — Chung kết: 3 seed, test đúng một lần mỗi seed

**DỪNG LẠI trước khi chạy ô dưới.** Chỉ chạy khi đã xong Bước 1-3 và **không còn muốn đổi cấu hình**. Ô này:
1. dựng dự đoán test của `F01` (3 seed) và của mốc `T00 + I00` (3 seed) bằng đúng một lượt test mỗi (exp, seed);
2. ghi `predictions/F01_seed*_test.csv`, `F01_uncal_*`, `F01rt_*`, `T00_*` và `F01_seed*_val.csv`;
3. gọi `eval.py score` và `eval.py grade` (không sửa `eval.py`), rồi phân tích lỗi.

Chạy lần hai sẽ bị từ chối (chốt chặn `test_started.json`). Không được dùng kết quả test để sửa cấu hình rồi chạy lại.

In [ ]:
CONFIRM_OPEN_TEST = False   # đổi thành True SAU KHI đã chốt cấu hình trên val
TEST_DONE = Path("predictions/F01_seed0_test.csv").exists()
if CONFIRM_OPEN_TEST and not TEST_DONE:
    !python {CODE_DIR}/final.py --images-dir {IMAGES_DIR} --labels-dir {LABELS_DIR} --out-dir runs --pred-dir predictions --yes-open-test
    TEST_DONE = Path("predictions/F01_seed0_test.csv").exists()
elif TEST_DONE:
    print("Test đã được mở ở phiên trước; không chạy lại.")
else:
    print("CHƯA mở test (CONFIRM_OPEN_TEST = False). Các ô chấm điểm bên dưới sẽ được bỏ qua.")

In [ ]:
# Chấm bằng eval.py gốc (đã viết sẵn trong repo). final.py đã chạy các lệnh này; ô này để xem lại kết quả.
if TEST_DONE:
    rt = json.load(open("runs/decisions.json", encoding="utf-8")).get("realtime") or {}
    lat = f"--latency-p95-ms {rt['p95_ms']:.3f} --latency-method proper" if rt.get("p95_ms") else ""
    !python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
    !python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out
    !python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" --baseline "predictions/T00_seed*_test.csv" --uncal "predictions/F01_uncal_seed*_test.csv" --final-val "predictions/F01_seed*_val.csv" --test-csv {LABELS_DIR}/test_subset0.csv --val-csv {LABELS_DIR}/val_subset0.csv --labels {LABELS_DIR}/labels.csv {lat} --out eval_out

In [ ]:
if TEST_DONE:
    display(Image("figures/confusion_test.png"))
    if Path("figures/errors_test.png").exists():
        display(Image("figures/errors_test.png"))
    display(pd.read_csv("eval_out/top_confusions.csv"))

## Bước 5 — Sản phẩm
`results.xlsx` (7 sheet: Backbones, Training, Inference, Final, PerClass, Latency, Summary) dựng từ log chạy thật và tính lại từ `predictions/` bằng `eval.py`; kiểm tra mỗi `exp_id` có ảnh trong `curves/` và số trong xlsx khớp. `report_tables.md` chứa các bảng markdown sinh tự động để dán vào `report.md`.

In [ ]:
import results as R
R.build_results("results.xlsx")
R.write_report_tables("report_tables.md")
problems = R.check_consistency("results.xlsx")
print("\n".join(problems) if problems else "OK: exp_id, curves, predictions, xlsx nhất quán")
pd.read_excel("results.xlsx", sheet_name="Summary")

In [ ]:
# Chép sản phẩm vào thư mục nộp bài. (.gitignore bỏ qua runs/ và eval_out/ nên lưu bản nhỏ vào logs/ và scores/.)
for name in ("curves", "predictions", "figures"):
    if Path(name).exists():
        shutil.copytree(name, SUB_DIR / name, dirs_exist_ok=True)
if Path("eval_out").exists():
    shutil.copytree("eval_out", SUB_DIR / "scores", dirs_exist_ok=True)
for name in ("results.xlsx", "report_tables.md"):
    shutil.copy(name, SUB_DIR / name)
shutil.copy("runs/decisions.json", SUB_DIR / "decisions.json")
# log nhỏ của từng lần chạy (truy ngược exp_id -> log, tiêu chí P2); KHÔNG chép .pt và logit lớn
for run in Path("runs").glob("*/seed*"):
    for f in ("config.json", "summary.json", "history.csv", "test_done.json"):
        if (run / f).exists():
            dst = SUB_DIR / "logs" / run.parent.name / run.name
            dst.mkdir(parents=True, exist_ok=True)
            shutil.copy(run / f, dst / f)
if Path("runs/inference").exists():
    shutil.copytree("runs/inference", SUB_DIR / "logs" / "inference", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.npz"))
print("Đã chép vào", SUB_DIR)
print("Tải thư mục", SUB_DIR, "về, điền report.md, README.md rồi nộp.")